# Regularized + Joint Luminance–Chrominance Fusion v2

**Purpose:** Evaluate frozen official RetinexFormer + HVI-CIDNet checkpoint pairs on LOL-v1, LOL-v2 Real, LOL-v2 Synthetic using **(a) individual models, (b) Train-fitted RGB scalar fusion, (c) unregularized YCrCb fusion, (d) dataset-specific regularized YCrCb fusion, (e) shared joint YCrCb fusion, (f) shared regularized joint YCrCb fusion**. No full retraining. No GT-mean. No x8 TTA.

**Train/Test:** Only original `Train` pairs are used for fitting and deterministic 3-fold *internal Train cross-validation* (no external Validation folder). Test is evaluated **once after all weights/lambdas are locked**. Cross-validation is used to choose lambda; no Test GT used for parameter fitting. However, original Train images may have been seen by the two pretrained teachers, and earlier project experiments have inspected official Test scores; this is not a completely blind evaluation.

**Interpretation:** `alpha_y` and `alpha_c` are RetinexFormer shares for Y and Cr/Cb, respectively. Joint weights are *shared across datasets*, but each dataset still has its own pretrained Retinex/HVI checkpoints; it is **not** an unseen-domain universal backbone. `regularized` weights are penalized toward **0.5** (symmetric neutral prior, not Test-informed). Joint optimization gives each dataset **equal macro weight**. Metrics and color transform follow the previous v1 notebook.

**Run:** Google Colab GPU (T4 or better), run cells top-to-bottom. Default is **all available original Train pairs** (485+689+900); to shorten a first pilot, set `FIT_TRAIN_COUNT=200` in Cell 3, and choose a new `RUN_NAME` for any change. Backbone inference dominates runtime. Re-running with unchanged settings reuses cached coefficient files. Test predictions and ZIP go to Google Drive. Full Test results only when `SMOKE_MAX_TEST=None`.

This repository edition imports the same reusable implementation as the Python CLIs.
The original notebook/report SHA-256 values and previously measured results are under
`reference_results/joint_lumachroma_v2/`. This notebook produces **new** outputs only when run.
Configure your storage roots below; no credentials or datasets belong in Git.

In [ ]:
# 1 — Install dependencies, mount storage, and select the implementation revision.
%pip -q install einops huggingface_hub hf_xet pandas scipy opencv-python-headless pillow tqdm matplotlib
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, subprocess, json
import torch

PROJECT = Path('/content/myRetinexformer_joint_v2')
# Set JOINT_FUSION_REVISION to a published commit hash for an immutable replay.
REVISION = os.environ.get('JOINT_FUSION_REVISION', 'feature/joint-regularized-lumachroma-fusion-v2')
def clone_at(url, folder, revision):
    if not (folder / '.git').exists():
        subprocess.run(['git', 'clone', '-q', url, str(folder)], check=True)
    subprocess.run(['git', '-C', str(folder), 'checkout', '-q', revision], check=True)
    return subprocess.check_output(['git', '-C', str(folder), 'rev-parse', 'HEAD'], text=True).strip()

IMPLEMENTATION_COMMIT = clone_at('https://github.com/Silly-Gooses/myRetinexformer.git', PROJECT, REVISION)
sys.path.insert(0, str(PROJECT))
assert torch.cuda.is_available(), 'Select a Colab GPU runtime.'
DEVICE = torch.device('cuda:0')
print('Implementation:', IMPLEMENTATION_COMMIT, 'GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2 — Exact official sources and configurable storage.
from fusion_model.joint_lumachroma.config import (
    RET_COMMIT, HVI_COMMIT, CV_FOLDS, SPLIT_SEED, LAMBDA_GRID, REGULARIZATION_CENTER, METHODS)
from fusion_model.joint_lumachroma.datasets import dataset_configs, discover_splits
from fusion_model.joint_lumachroma.model_loader import resolve_weights, verify_source
from fusion_model.joint_lumachroma.train_coefficients import fit_experiment
from fusion_model.joint_lumachroma.evaluation import evaluate_experiment

STORAGE = Path(os.environ.get('JOINT_FUSION_STORAGE', '/content/drive/MyDrive'))
DATA = Path(os.environ.get('JOINT_FUSION_DATA', str(STORAGE / 'datasets')))
RET_WEIGHTS = Path(os.environ.get('JOINT_FUSION_RET_WEIGHTS', str(STORAGE / 'RetinexFormer_Official_Pretrained')))
HVI_WEIGHTS = Path(os.environ.get('JOINT_FUSION_HVI_WEIGHTS', str(STORAGE / 'HVI_CIDNet_Official_Weights' / 'weights')))
RUN_NAME = 'regjoint_alltrain_v2'
OUT = Path(os.environ.get('JOINT_FUSION_OUTPUT', str(STORAGE / 'joint-fusion-runs' / RUN_NAME)))
SELECTED_DATASETS = ['LOLv1', 'LOLv2_Real', 'LOLv2_Synthetic']
FIT_TRAIN_COUNT = None  # all 485 + 689 + 900 original Train pairs
SMOKE_MAX_TEST = None  # integer = smoke only, never a full reference reproduction
SAVE_ALL_TEST_PREDICTIONS = True
PREVIEW_COUNT = 3
CFG = dataset_configs(DATA)
RET_REPO = Path('/content/Retinexformer_ds_ycc_joint_v2')
HVI_REPO = Path('/content/HVI_CIDNet_ds_ycc_joint_v2')
assert clone_at('https://github.com/caiyuanhao1998/Retinexformer.git', RET_REPO, RET_COMMIT) == RET_COMMIT
assert clone_at('https://github.com/Fediory/HVI-CIDNet.git', HVI_REPO, HVI_COMMIT) == HVI_COMMIT
verify_source(RET_REPO, RET_COMMIT, 'basicsr/models/archs')
verify_source(HVI_REPO, HVI_COMMIT, 'net')
print('Output:', OUT)

In [ ]:
# 3 — Validate exact pairing (filenames only for Test) and checkpoint hashes.
SPLITS = discover_splits(CFG, SELECTED_DATASETS, FIT_TRAIN_COUNT)
WEIGHTS = resolve_weights(SELECTED_DATASETS, RET_WEIGHTS, HVI_WEIGHTS)
for ds in SELECTED_DATASETS:
    print(ds, 'Train fit:', len(SPLITS[ds]['fit']), 'Test:', len(SPLITS[ds]['test']))
    print('RET SHA256:', WEIGHTS[ds]['ret_sha256'], 'HVI SHA256:', WEIGHTS[ds]['hvi_sha256'])

## 4 — Train-only coefficients, cross-validation, and locked weights

The fitter caches per-image quadratic coefficients, performs local and macro-balanced joint
three-fold CV, and freezes all five ablations before Test. It preserves the seven starts,
lambda grid, 0.005 dB stronger-regularization tie rule, and unclipped affine fitting objective.
Changed protocol settings require a new output directory. The full-precision reference
weights are not substituted for this fit.

In [ ]:
params = fit_experiment(CFG, SELECTED_DATASETS, WEIGHTS, RET_REPO, HVI_REPO,
                        DEVICE, OUT, FIT_TRAIN_COUNT)
print(json.dumps(params, indent=2))
print('ALL WEIGHTS SELECTED AND FROZEN. Test pixels/GT have not been used.')

## 5 — Evaluate all seven methods with frozen weights

This stage does not fit or select weights. RetinexFormer RGB8 uses rounding; HVI and
fusion RGB8 use truncation. Float RGB PSNR and saved-RGB8 PSNR/SSIM remain separately labeled.

In [ ]:
summary = evaluate_experiment(CFG, SELECTED_DATASETS, WEIGHTS, RET_REPO, HVI_REPO,
                              DEVICE, OUT, OUT / 'selected_weights_train_only.json',
                              SMOKE_MAX_TEST, SAVE_ALL_TEST_PREDICTIONS, PREVIEW_COUNT)
display(summary[['dataset'] + [m + '_psnr_float' for m in METHODS]])
display(summary[['dataset'] + [m + '_ssim_rgb8' for m in METHODS]])
print('SMOKE ONLY' if SMOKE_MAX_TEST is not None else 'FULL OFFICIAL TEST RUN')
print('Report:', OUT / 'RetinexFormer_HVI_RegularizedJoint_LumaChroma_Report.zip')

In [ ]:
# 6 — Visual comparisons from the first saved example in each dataset.
from PIL import Image
import matplotlib.pyplot as plt
for ds in SELECTED_DATASETS:
    examples = sorted((OUT / ds / 'previews').glob('*_low.png'))
    if not examples:
        continue
    stem = examples[0].name.removesuffix('_low.png')
    names = ['low', 'ret', 'hvi', 'ycc_joint_regularized', 'gt']
    fig, axes = plt.subplots(1, len(names), figsize=(20, 5))
    for ax, name in zip(axes, names):
        with Image.open(OUT / ds / 'previews' / (stem + '_' + name + '.png')) as image:
            ax.imshow(image)
        ax.set_title(name); ax.axis('off')
    fig.suptitle(ds)
    plt.show()

## What to report

- **RetinexFormer/HVI-only:** checkpoint-specific standalone anchors.
- **RGB fit:** one scalar weight per dataset, estimated from Train.
- **YCrCb fit:** two weights (Y, Cr/Cb) per dataset, unregularized.
- **YCrCb regularized:** two dataset-specific weights with lambda chosen by Train-fold CV.
- **YCrCb joint:** one pair of weights fitted across all three Train datasets (equal domain weighting).
- **YCrCb joint regularized:** shared pair of weights with lambda selected using joint Train-fold CV.

`train_cross_validation_summary.csv` explains selected lambdas; `selected_weights_train_only.json` locks them; `test_summary.csv` gives official comparison under one consistent metric. **Don't select the best Test method retrospectively and call it an untouched-test improvement.** Test-set follow-up variants should be labeled exploratory. For formal generalization evidence, additionally test on a new holdout domain or independently collected images.